<a href="https://colab.research.google.com/github/kpdhanya-lab/Dracula-Backup/blob/main/FLOODCAST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/kratzert/Caravan.git



Cloning into 'Caravan'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 255 (delta 113), reused 95 (delta 95), pack-reused 121 (from 1)
Receiving objects: 100% (255/255), 4.04 MiB | 24.75 MiB/s, done.
Resolving deltas: 100% (137/137), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Create a dedicated folder for the challenge in your Google Drive
project_dir = '/content/drive/My Drive/BASINSHIFT_Challenge'
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

print(f"Working directory set to: {project_dir}")


Working directory set to: /content/drive/My Drive/BASINSHIFT_Challenge


In [ ]:
# Example command to download Caravan CSV archive directly to your mounted Drive folder
!wget -O Caravan-csv.tar.xz https://zenodo.org/records/14673536/files/Caravan-csv.tar.xz

--2026-11-04 16:55:22--  https://zenodo.org/records/14673536/files/Caravan-csv.tar.xz
Resolving zenodo.org (zenodo.org)... 188.185.51.60, 137.138.121.78, 188.185.42.113, ...
Connecting to zenodo.org (zenodo.org)|188.185.51.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22097154380 (21G) [application/octet-stream]
Saving to: ‘Caravan-csv.tar.xz’

Caravan-csv.tar.xz    0%[                    ]  29.91M   768KB/s    eta 8h 35m ^C


In [ ]:
!apt-get install aria2 -y -qq

Selecting previously unselected package libcares2:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1.1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1.1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1.1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a sy

In [ ]:
!aria2c -c -x 8 -s 8 -k 1M --file-allocation=none -o Caravan-csv.tar.xz "https://zenodo.org/api/records/14673536/files/Caravan-csv.tar.xz/content"


11/04 16:59:24 [NOTICE] Downloading 1 item(s)
 *** Download Progress Summary as of Wed Nov  4 17:00:25 2026 *** 
=
[#8dcbe7 555MiB/20GiB(2%) CN:8 DL:9.8MiB ETA:34m53s]
FILE: /content/drive/MyDrive/BASINSHIFT_Challenge/Caravan-csv.tar.xz
-

 *** Download Progress Summary as of Wed Nov  4 17:01:25 2026 *** 
=
[#8dcbe7 1.1GiB/20GiB(5%) CN:8 DL:10MiB ETA:32m5s]
FILE: /content/drive/MyDrive/BASINSHIFT_Challenge/Caravan-csv.tar.xz
-

 *** Download Progress Summary as of Wed Nov  4 17:02:26 2026 *** 
=
[#8dcbe7 1.7GiB/20GiB(8%) CN:8 DL:10MiB ETA:30m15s]
FILE: /content/drive/MyDrive/BASINSHIFT_Challenge/Caravan-csv.tar.xz
-

 *** Download Progress Summary as of Wed Nov  4 17:03:27 2026 *** 
=
[#8dcbe7 2.3GiB/20GiB(11%) CN:8 DL:9.3MiB ETA:33m21s]
FILE: /content/drive/MyDrive/BASINSHIFT_Challenge/Caravan-csv.tar.xz
-

 *** Download Progress Summary as of Wed Nov  4 17:04:27 2026 *** 
=
[#8dcbe7 2.9GiB/20GiB(14%) CN:8 DL:7.2MiB ETA:41m49s]
FILE: /content/drive/MyDrive/BASINSHIFT_Challenge/Carava

In [ ]:
import os

# 1. Create a temporary folder on Colab's local high-speed SSD
local_dir = '/content/caravan_local'
os.makedirs(local_dir, exist_ok=True)

# 2. Extract the archive there (this will run much faster than extracting to Google Drive)
!tar -xf /content/drive/My\ Drive/BASINSHIFT_Challenge/Caravan-csv.tar.xz -C {local_dir}

print("Extraction complete on local SSD!")

^C
Extraction complete on local SSD!


In [ ]:
!du -sh /content/caravan_local

19G	/content/caravan_local


In [ ]:
import os
import glob
import json
import random

# Scan the local SSD directory for all catchment CSVs
local_dir = '/content/caravan_local'
all_catchment_files = glob.glob(os.path.join(local_dir, '**', '*.csv'), recursive=True)

# Extract unique catchment IDs from filenames
catchment_ids = [os.path.basename(f).replace('.csv', '') for f in all_catchment_files]
print(f"Successfully loaded {len(catchment_ids)} catchments from local SSD!")

# Create the 60/20/20 split manifest
def create_basinshift_splits(ids_list, train_ratio=0.60, val_ratio=0.20, test_ratio=0.20, seed=42):
    assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-5
    random.seed(seed)
    shuffled_ids = list(ids_list)
    random.shuffle(shuffled_ids)

    n_total = len(shuffled_ids)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)

    return {
        "train": shuffled_ids[:n_train],
        "val": shuffled_ids[n_train:n_train + n_val],
        "test": shuffled_ids[n_train + n_val:]
    }

basin_splits = create_basinshift_splits(catchment_ids)

# Save manifest to Google Drive for persistence
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(basin_splits, f, indent=4)

print(f"Split manifest saved safely to Google Drive: {manifest_path}")

Successfully loaded 3428 catchments from local SSD!
Split manifest saved safely to Google Drive: /content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json


In [ ]:
import polars as pl
import numpy as np

def load_and_clean_catchment(file_path):
    """
    Loads a catchment CSV file using Polars, downcasts floats to float32 to save RAM,
    and adheres to missing-value governance rules.
    """
    # Lazy or direct read with Polars (blazing fast)
    df = pl.read_csv(file_path)

    # Downcast float64 columns to float32 to cut memory usage in half
    float_cols = [col for col, dtype in df.schema.items() if dtype == pl.Float64]
    if float_cols:
        df = df.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols])

    return df

# Example test: load the first catchment file from your local directory to check schema
import glob
sample_file = glob.glob('/content/caravan_local/**/*.csv', recursive=True)[0]
sample_df = load_and_clean_catchment(sample_file)

print(f"Sample catchment loaded successfully from: {sample_file}")
print(sample_df.head(3))

Sample catchment loaded successfully from: /content/caravan_local/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv/timeseries/csv/hysets/hysets_04LE002.csv
shape: (3, 41)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ date      ┆ dewpoint_ ┆ dewpoint_ ┆ dewpoint_ ┆ … ┆ volumetri ┆ volumetri ┆ volumetri ┆ volumetr │
│ ---       ┆ temperatu ┆ temperatu ┆ temperatu ┆   ┆ c_soil_wa ┆ c_soil_wa ┆ c_soil_wa ┆ ic_soil_ │
│ str       ┆ re_2m_max ┆ re_2m_mea ┆ re_2m_min ┆   ┆ ter_layer ┆ ter_layer ┆ ter_layer ┆ water_la │
│           ┆ ---       ┆ n         ┆ ---       ┆   ┆ _3_…      ┆ _4_…      ┆ _4_…      ┆ yer_4_…  │
│           ┆ f32       ┆ ---       ┆ f32       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ f32       ┆           ┆   ┆ f32       ┆ f32       ┆ f32       ┆ f32      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1951-01-0 ┆ -1

In [ ]:
import json
import glob
import os
import polars as pl
import numpy as np

# 1. Load your split manifest from Google Drive
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
    splits = json.load(f)

train_ids = set(splits['train'])
val_ids = set(splits['val'])
test_ids = set(splits['test'])

print(f"Loaded splits -> Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

# 2. Map catchment IDs to their local file paths
local_dir = '/content/caravan_local'
all_files = glob.glob(os.path.join(local_dir, '**', '*.csv'), recursive=True)
file_map = {os.path.basename(f).replace('.csv', ''): f for f in all_files}

# 3. Quick sanity check: verify we can isolate training files
sample_train_file = file_map[list(train_ids)[0]]
print(f"Verified training file mapping: {sample_train_file}")

Loaded splits -> Train: 2056, Val: 685, Test: 687
Verified training file mapping: /content/caravan_local/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv/timeseries/csv/camels/camels_12010000.csv


In [ ]:
import numpy as np
import polars as pl
import polars.selectors as cs

def compute_train_statistics(train_ids, file_map, sample_size=50):
    """
    Computes mean and std strictly for numeric feature columns across a sample
    of training catchments, ensuring shape alignment to avoid broadcasting errors.
    """
    subset_ids = list(train_ids)[:sample_size]
    feature_sums = None
    feature_sq_sums = None
    total_count = 0
    common_features = None

    for cid in subset_ids:
        fpath = file_map.get(cid)
        if not fpath:
            continue
        df = pl.read_csv(fpath)
        num_df = df.select(cs.numeric())

        # Initialize or lock in the common feature columns based on the first file
        if common_features is None:
            common_features = num_df.columns
        else:
            # Ensure current file has all common columns, select only those
            missing_cols = [c for c in common_features if c not in num_df.columns]
            if missing_cols:
                continue # Skip catchments missing standard columns
            num_df = num_df.select(common_features)

        curr_sum = num_df.sum().to_numpy()[0]
        curr_sq_sum = num_df.select([pl.col(c).pow(2) for c in common_features]).sum().to_numpy()[0]

        if feature_sums is None:
            feature_sums = curr_sum
            feature_sq_sums = curr_sq_sum
        else:
            feature_sums += curr_sum
            feature_sq_sums += curr_sq_sum

        total_count += len(num_df)

    mean = feature_sums / total_count
    std = np.sqrt((feature_sq_sums / total_count) - (mean ** 2))
    std = np.where(std == 0, 1.0, std) # Prevent division by zero

    return mean, std, common_features

# Run calculation again
train_means, train_stds, feature_names = compute_train_statistics(train_ids, file_map)
print(f"Computed normalization stats successfully for {len(feature_names)} standardized numeric features!")

Computed normalization stats successfully for 39 standardized numeric features!


In [ ]:
import os
import glob
import polars as pl
import polars.selectors as cs

def cache_catchments_to_parquet(file_map, output_dir='/content/caravan_parquet'):
    """
    Converts raw catchment CSVs into compressed Parquet files for ultra-fast loading.
    """
    os.makedirs(output_dir, exist_ok=True)

    print(f"Starting Parquet caching for {len(file_map)} catchments...")
    success_count = 0

    for cid, fpath in file_map.items():
        try:
            # Read CSV and downcast/clean
            df = pl.read_csv(fpath)

            # Downcast float64 to float32
            float_cols = [col for col, dtype in df.schema.items() if dtype == pl.Float64]
            if float_cols:
                df = df.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols])

            # Define output path
            out_path = os.path.join(output_dir, f"{cid}.parquet")

            # Write to compressed Parquet
            df.write_parquet(out_path, compression="snappy")
            success_count += 1

            if success_count % 500 == 0:
                print(f"Cached {success_count}/{len(file_map)} catchments...")
        except Exception as e:
            print(f"Error caching {cid}: {e}")

    print(f"Parquet caching complete! Successfully processed {success_count} catchments.")
    return output_dir

# Execute caching
parquet_dir = cache_catchments_to_parquet(file_map)

Starting Parquet caching for 3428 catchments...
Cached 500/3428 catchments...
Cached 1000/3428 catchments...
Cached 1500/3428 catchments...
Cached 2000/3428 catchments...
Cached 2500/3428 catchments...
Cached 3000/3428 catchments...
Parquet caching complete! Successfully processed 3428 catchments.


In [ ]:
import json
import os
import polars as pl
import polars.selectors as cs
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Load split manifest
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
    splits = json.load(f)

train_ids = set(splits['train'])
val_ids = set(splits['val'])

parquet_dir = '/content/caravan_parquet'

def prepare_data_for_lgb(catchment_ids, parquet_dir, max_catchments=20):
    """
    Loads a subset of catchments from Parquet using Polars selectors,
    and concatenates them into a single training/validation matrix.
    """
    X_list, y_list = [], []
    count = 0

    for cid in catchment_ids:
        if count >= max_catchments:
            break
        fpath = os.path.join(parquet_dir, f"{cid}.parquet")
        if not os.path.exists(fpath):
            continue

        df = pl.read_parquet(fpath)

        # Dynamically find the target column for streamflow/discharge
        target_cols = [c for c in df.columns if 'discharge' in c.lower() or 'streamflow' in c.lower()]
        if not target_cols:
            continue
        target_col = target_cols[0]

        # Drop nulls on target
        df_clean = df.drop_nulls(subset=[target_col])
        if len(df_clean) < 100:
            continue

        # Select numeric features excluding the target column using polars.selectors
        numeric_df = df_clean.select(cs.numeric())
        feature_cols = [c for c in numeric_df.columns if c != target_col]

        X = df_clean.select(feature_cols).to_numpy()
        y = df_clean.select(target_col).to_numpy().ravel()

        X_list.append(X)
        y_list.append(y)
        count += 1

    if not X_list:
        raise ValueError("No valid data loaded. Check target column names.")

    return np.vstack(X_list), np.concatenate(y_list)

print("Preparing training data matrix...")
X_train, y_train = prepare_data_for_lgb(list(train_ids), parquet_dir, max_catchments=20)
X_val, y_val = prepare_data_for_lgb(list(val_ids), parquet_dir, max_catchments=5)

print(f"Training Matrix Shape: {X_train.shape}")
print(f"Validation Matrix Shape: {X_val.shape}")

# 2. Train LightGBM Baseline Model
print("Training LightGBM baseline model...")
model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=10)]
)

# 3. Evaluate Baseline
preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
mae = mean_absolute_error(y_val, preds)

print("--- Baseline Results ---")
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE:  {mae:.4f}")

Preparing training data matrix...
Training Matrix Shape: (225684, 39)
Validation Matrix Shape: (61765, 39)
Training LightGBM baseline model...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.098868 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7104
[LightGBM] [Info] Number of data points in the train set: 225684, number of used features: 38
[LightGBM] [Info] Start training from score 1.845575
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[39]	valid_0's l2: 5.1601


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Baseline Results ---
Validation RMSE: 2.2716
Validation MAE:  1.3639


In [ ]:
# Add lag features for key meteorological variables (e.g., precipitation, temperature)
# You can use Polars shift to create lags efficiently:
df_lagged = df_clean.with_columns([
    pl.col("total_precipitation_sum").shift(1).alias("precip_lag_1"),
    pl.col("total_precipitation_sum").rolling_mean(window_size=7).alias("precip_roll_7"),
])

NameError: name 'df_clean' is not defined

In [ ]:
import json
import os
import polars as pl
import polars.selectors as cs
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Load split manifest
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
    splits = json.load(f)

train_ids = set(splits['train'])
val_ids = set(splits['val'])
parquet_dir = '/content/caravan_parquet'

def prepare_data_with_lags(catchment_ids, parquet_dir, max_catchments=50):
    """
    Loads Parquet catchments, ensures numeric casting, adds hydrological lag/rolling features,
    and returns feature matrix and numeric target array.
    """
    X_list, y_list = [], []
    count = 0

    for cid in catchment_ids:
        if count >= max_catchments:
            break
        fpath = os.path.join(parquet_dir, f"{cid}.parquet")
        if not os.path.exists(fpath):
            continue

        df = pl.read_parquet(fpath)

        # Dynamically find streamflow target column
        target_cols = [c for c in df.columns if 'discharge' in c.lower() or 'streamflow' in c.lower()]
        if not target_cols:
            continue
        target_col = target_cols[0]

        # Ensure target column is strictly cast to Float32, dropping unparseable rows
        df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))

        # Drop nulls on target and sort chronologically if date exists
        df_clean = df.drop_nulls(subset=[target_col])
        if 'date' in df_clean.columns:
            df_clean = df_clean.sort('date')

        if len(df_clean) < 100:
            continue

        # Feature Engineering: Add rolling / lag features if precipitation exists
        precip_col = next((c for c in df_clean.columns if 'precip' in c.lower()), None)
        if precip_col:
            df_clean = df_clean.with_columns([
                pl.col(precip_col).shift(1).alias("precip_lag_1"),
                pl.col(precip_col).rolling_mean(window_size=7).alias("precip_roll_7")
            ])

        # Drop null rows created by shifting/rolling
        df_clean = df_clean.drop_nulls()

        # Select numeric features excluding target and date
        numeric_df = df_clean.select(cs.numeric())
        feature_cols = [c for c in numeric_df.columns if c != target_col]

        X = df_clean.select(feature_cols).to_numpy()
        y = df_clean.select(target_col).to_numpy().ravel()

        # Final safety check to ensure y is purely numeric float data
        if not np.issubdtype(y.dtype, np.number):
            continue

        X_list.append(X)
        y_list.append(y)
        count += 1

    if not X_list:
        raise ValueError("No valid data loaded. Check columns.")

    return np.vstack(X_list), np.concatenate(y_list)

print("Preparing training data with lag features...")
X_train, y_train = prepare_data_with_lags(list(train_ids), parquet_dir, max_catchments=50)
X_val, y_val = prepare_data_with_lags(list(val_ids), parquet_dir, max_catchments=15)

print(f"Training Matrix Shape: {X_train.shape}")
print(f"Validation Matrix Shape: {X_val.shape}")

# Apply log1p transformation safely now that targets are guaranteed numeric floats
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

# 2. Train Tuned LightGBM Baseline Model
print("Training tuned LightGBM model...")
model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=63,
    feature_fraction=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=[lgb.early_stopping(stopping_rounds=15)]
)

# 3. Evaluate Model
preds_log = model.predict(X_val)
preds = np.expm1(preds_log)

rmse = np.sqrt(mean_squared_error(y_val, preds))
mae = mean_absolute_error(y_val, preds)

print("--- Improved Baseline Results ---")
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE:  {mae:.4f}")

Preparing training data with lag features...
Training Matrix Shape: (552475, 41)
Validation Matrix Shape: (164949, 41)
Training tuned LightGBM model...
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.473768 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7823
[LightGBM] [Info] Number of data points in the train set: 552475, number of used features: 40
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Start training from score 0.745362
Training until validation scores don't improve for 15 rounds
Early stopping, best iteration is:
[94]	valid_0's l2: 0.179004
[LightGBM

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Improved Baseline Results ---
Validation RMSE: 2.2093
Validation MAE:  0.8030


In [ ]:
import json
import os
import polars as pl
import polars.selectors as cs
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Load split manifest
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
    splits = json.load(f)

train_ids = set(splits['train'])
val_ids = set(splits['val'])
parquet_dir = '/content/caravan_parquet'

def prepare_data_with_feature_names(catchment_ids, parquet_dir, max_catchments=50):
    """
    Loads Parquet catchments and returns feature matrices as Polars DataFrames
    to preserve feature names and avoid sklearn/LightGBM warnings.
    """
    df_list = []
    count = 0

    for cid in catchment_ids:
        if count >= max_catchments:
            break
        fpath = os.path.join(parquet_dir, f"{cid}.parquet")
        if not os.path.exists(fpath):
            continue

        df = pl.read_parquet(fpath)

        # Dynamically find streamflow target column
        target_cols = [c for c in df.columns if 'discharge' in c.lower() or 'streamflow' in c.lower()]
        if not target_cols:
            continue
        target_col = target_cols[0]

        # Ensure target column is strictly numeric
        df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
        df_clean = df.drop_nulls(subset=[target_col])

        if 'date' in df_clean.columns:
            df_clean = df_clean.sort('date')

        if len(df_clean) < 100:
            continue

        # Add rolling / lag features if precipitation exists
        precip_col = next((c for c in df_clean.columns if 'precip' in c.lower()), None)
        if precip_col:
            df_clean = df_clean.with_columns([
                pl.col(precip_col).shift(1).alias("precip_lag_1"),
                pl.col(precip_col).rolling_mean(window_size=7).alias("precip_roll_7")
            ])

        df_clean = df_clean.drop_nulls()
        if len(df_clean) == 0:
            continue

        df_list.append(df_clean)
        count += 1

    if not df_list:
        raise ValueError("No valid data loaded.")

    # Concatenate all catchment dataframes vertically
    combined_df = pl.concat(df_list, how="vertical_relaxed")

    # Identify target and feature columns
    target_cols = [c for c in combined_df.columns if 'discharge' in c.lower() or 'streamflow' in c.lower()]
    target_col = target_cols[0]

    numeric_cols = combined_df.select(cs.numeric()).columns
    feature_cols = [c for c in numeric_cols if c != target_col]

    # Convert to Pandas DataFrames/Series to retain feature names cleanly
    pdf = combined_df.select(feature_cols + [target_col]).to_pandas()

    X = pdf[feature_cols]
    y = pdf[target_col].values

    return X, y

print("Preparing training data with feature names...")
X_train, y_train = prepare_data_with_feature_names(list(train_ids), parquet_dir, max_catchments=50)
X_val, y_val = prepare_data_with_feature_names(list(val_ids), parquet_dir, max_catchments=15)

print(f"Training Matrix Shape: {X_train.shape}")
print(f"Validation Matrix Shape: {X_val.shape}")

# Keep targets on original scale or use milder transform,
# and use Huber objective to handle flood outliers without exploding RMSE.
y_train_target = y_train
y_val_target = y_val

# 2. Train LightGBM with Huber Loss (Robust to outliers/floods)
print("Training robust LightGBM model with Huber loss...")
model = lgb.LGBMRegressor(
    objective='huber',          # Robust to extreme discharge/flood peaks
    alpha=0.9,                  # Huber loss parameter
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=63,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train_target,
    eval_set=[(X_val, y_val_target)],
    callbacks=[lgb.early_stopping(stopping_rounds=15)]
)

# 3. Evaluate Model
preds = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val_target, preds))
mae = mean_absolute_error(y_val_target, preds)

print("--- Robust Baseline Results ---")
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE:  {mae:.4f}")

Preparing training data with feature names...
Training Matrix Shape: (552475, 41)
Validation Matrix Shape: (164949, 41)
Training robust LightGBM model with Huber loss...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.454971 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7823
[LightGBM] [Info] Number of data points in the train set: 552475, number of used features: 40
[LightGBM] [Info] Start training from score 1.970018
Training until validation scores don't improve for 15 rounds
Early stopping, best iteration is:
[301]	valid_0's huber: 0.478678
--- Robust Baseline Results ---
Validation RMSE: 2.4036
Validation MAE:  0.8424


In [ ]:
import json
import os
import lightgbm as lgb
import numpy as np
import polars as pl
import polars.selectors as cs
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load split manifest (preserving your 60/20/20 geographic partition)
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
  splits = json.load(f)

train_ids = set(splits['train'])
val_ids = set(splits['val'])
parquet_dir = '/content/caravan_parquet'


def prepare_floodcast_data(catchment_ids, parquet_dir, max_catchments=60):
  """Loads Parquet catchments, adds historical streamflow and precipitation lags

  for robust flood memory, and returns feature matrices as Pandas DataFrames.
  """
  df_list = []
  count = 0

  for cid in catchment_ids:
    if count >= max_catchments:
      break
    fpath = os.path.join(parquet_dir, f'{cid}.parquet')
    if not os.path.exists(fpath):
      continue

    df = pl.read_parquet(fpath)

    # Dynamically find streamflow target column
    target_cols = [
        c
        for c in df.columns
        if 'discharge' in c.lower() or 'streamflow' in c.lower()
    ]
    if not target_cols:
      continue
    target_col = target_cols[0]

    # Ensure numeric casting
    df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
    df_clean = df.drop_nulls(subset=[target_col])

    if 'date' in df_clean.columns:
      df_clean = df_clean.sort('date')

    if len(df_clean) < 100:
      continue

    # FEATURE ENGINEERING FOR FLOODS:
    # 1. Historical Streamflow Lags (Crucial for routing & groundwater memory)
    # 2. Meteorological Precipitation Lags & Rolling windows
    precip_col = next(
        (c for c in df_clean.columns if 'precip' in c.lower()), None
    )

    lag_expressions = [
        pl.col(target_col).shift(1).alias('streamflow_lag_1'),
        pl.col(target_col).rolling_mean(window_size=3).alias('streamflow_roll_3'),
    ]

    if precip_col:
      lag_expressions.extend([
          pl.col(precip_col).shift(1).alias('precip_lag_1'),
          pl.col(precip_col).rolling_mean(window_size=7).alias('precip_roll_7'),
      ])

    df_clean = df_clean.with_columns(lag_expressions).drop_nulls()

    if len(df_clean) == 0:
      continue

    df_list.append(df_clean)
    count += 1

  if not df_list:
    raise ValueError('No valid data loaded.')

  # Concatenate efficiently
  combined_df = pl.concat(df_list, how='vertical_relaxed')

  target_cols = [
      c
      for c in combined_df.columns
      if 'discharge' in c.lower() or 'streamflow' in c.lower()
  ]
  target_col = target_cols[0]

  numeric_cols = combined_df.select(cs.numeric()).columns
  feature_cols = [c for c in numeric_cols if c != target_col]

  # Convert to Pandas to preserve feature names cleanly for LightGBM
  pdf = combined_df.select(feature_cols + [target_col]).to_pandas()

  return pdf[feature_cols], pdf[target_col].values


print('Preparing training data with advanced hydrological features...')
# Scale up catchment count slightly for better cross-catchment generalization
X_train, y_train = prepare_floodcast_data(
    list(train_ids), parquet_dir, max_catchments=80
)
X_val, y_val = prepare_floodcast_data(
    list(val_ids), parquet_dir, max_catchments=25
)

print(f'Training Matrix Shape: {X_train.shape}')
print(f'Validation Matrix Shape: {X_val.shape}')

# 2. Train LightGBM Model with Your Recommended Configuration
print(
    'Training tuned LightGBM model with regression objective & regularization...'
)
model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=8,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50),
    ],
)

# 3. Evaluation
preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
mae = mean_absolute_error(y_val, preds)

print('--- Optimized FloodCast Results ---')
print(f'Validation RMSE: {rmse:.4f}')
print(f'Validation MAE:  {mae:.4f}')

Preparing training data with advanced hydrological features...
Training Matrix Shape: (889211, 43)
Validation Matrix Shape: (290711, 43)
Training tuned LightGBM model with regression objective & regularization...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.786412 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8333
[LightGBM] [Info] Number of data points in the train set: 889211, number of used features: 42
[LightGBM] [Info] Start training from score 1.747796
Training until validation scores don't improve for 50 rounds
[50]	valid_0's l2: 1.33512
[100]	valid_0's l2: 0.844092
[150]	valid_0's l2: 0.757509
[200]	valid_0's l2: 0.722036
[250]	valid_0's l2: 0.704418
[300]	valid_0's l2: 0.694441
[350]	valid_0's l2: 0.686387
[400]	valid_0's l2: 0.68157
[450]	valid_0's l2: 0.681053
[500]	valid_0's l2: 0.679752
[550]	valid_0's l2: 0.67929
Early stopping, best iteration is:
[509]	valid_0's l2: 0.678471
--- 

In [ ]:
import json
import os
import lightgbm as lgb
import numpy as np
import polars as pl
import polars.selectors as cs
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load split manifest
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
  splits = json.load(f)

train_ids = set(splits['train'])
val_ids = set(splits['val'])
parquet_dir = '/content/caravan_parquet'


def prepare_advanced_floodcast_data(
    catchment_ids, parquet_dir, max_catchments=100
):
  """Loads Parquet catchments and computes advanced multi-scale hydrological

  lags, rolling sums, and soil saturation windows.
  """
  df_list = []
  count = 0

  for cid in catchment_ids:
    if count >= max_catchments:
      break
    fpath = os.path.join(parquet_dir, f'{cid}.parquet')
    if not os.path.exists(fpath):
      continue

    df = pl.read_parquet(fpath)

    target_cols = [
        c
        for c in df.columns
        if 'discharge' in c.lower() or 'streamflow' in c.lower()
    ]
    if not target_cols:
      continue
    target_col = target_cols[0]

    df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
    df_clean = df.drop_nulls(subset=[target_col])

    if 'date' in df_clean.columns:
      df_clean = df_clean.sort('date')

    if len(df_clean) < 100:
      continue

    precip_col = next(
        (c for c in df_clean.columns if 'precip' in c.lower()), None
    )

    # ADVANCED HYDROLOGICAL FEATURE ENGINEERING
    lag_expressions = [
        pl.col(target_col).shift(1).alias('streamflow_lag_1'),
        pl.col(target_col).shift(2).alias('streamflow_lag_2'),
        pl.col(target_col).shift(7).alias('streamflow_lag_7'),
        pl.col(target_col).rolling_mean(window_size=3).alias('streamflow_roll_3'),
        pl.col(target_col).rolling_mean(window_size=7).alias('streamflow_roll_7'),
    ]

    if precip_col:
      lag_expressions.extend([
          pl.col(precip_col).shift(1).alias('precip_lag_1'),
          pl.col(precip_col).rolling_mean(window_size=7).alias('precip_roll_7'),
          # Long-term antecedent moisture conditions (crucial for floods)
          pl.col(precip_col)
          .rolling_mean(window_size=14)
          .alias('precip_roll_14'),
          pl.col(precip_col)
          .rolling_mean(window_size=30)
          .alias('precip_roll_30'),
          # Storm intensity / volatility
          pl.col(precip_col).rolling_std(window_size=7).alias('precip_std_7'),
      ])

    df_clean = df_clean.with_columns(lag_expressions).drop_nulls()
    if len(df_clean) == 0:
      continue

    df_list.append(df_clean)
    count += 1

  if not df_list:
    raise ValueError('No valid data loaded.')

  combined_df = pl.concat(df_list, how='vertical_relaxed')

  target_cols = [
      c
      for c in combined_df.columns
      if 'discharge' in c.lower() or 'streamflow' in c.lower()
  ]
  target_col = target_cols[0]

  numeric_cols = combined_df.select(cs.numeric()).columns
  feature_cols = [c for c in numeric_cols if c != target_col]

  pdf = combined_df.select(feature_cols + [target_col]).to_pandas()
  return pdf[feature_cols], pdf[target_col].values


print('Preparing training data with advanced hydrological memory features...')
X_train, y_train = prepare_advanced_floodcast_data(
    list(train_ids), parquet_dir, max_catchments=100
)
X_val, y_val = prepare_advanced_floodcast_data(
    list(val_ids), parquet_dir, max_catchments=35
)

print(f'Training Matrix Shape: {X_train.shape}')
print(f'Validation Matrix Shape: {X_val.shape}')

# Log-transform target to handle extreme flood skew
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

print('Training enhanced LightGBM model with extended feature set...')
model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=2500,
    learning_rate=0.02,
    num_leaves=63,
    max_depth=9,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=75),
        lgb.log_evaluation(period=100),
    ],
)

# Evaluation
preds_log = model.predict(X_val)
preds = np.expm1(preds_log)

rmse = np.sqrt(mean_squared_error(y_val, preds))
mae = mean_absolute_error(y_val, preds)

print('--- Advanced Feature FloodCast Results ---')
print(f'Validation RMSE: {rmse:.4f}')
print(f'Validation MAE:  {mae:.4f}')

Preparing training data with advanced hydrological memory features...
Training Matrix Shape: (1109345, 49)
Validation Matrix Shape: (427095, 49)
Training enhanced LightGBM model with extended feature set...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.532945 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9862
[LightGBM] [Info] Number of data points in the train set: 1109345, number of used features: 48
[LightGBM] [Info] Start training from score 0.697813
Training until validation scores don't improve for 75 rounds
[100]	valid_0's l2: 0.0113024
[200]	valid_0's l2: 0.00251038
[300]	valid_0's l2: 0.00179019
[400]	valid_0's l2: 0.00153211
[500]	valid_0's l2: 0.00138862
[600]	valid_0's l2: 0.00129155
[700]	valid_0's l2: 0.00120732
[800]	valid_0's l2: 0.0011486
[900]	valid_0's l2: 0.00109842
[1000]	valid_0's l2: 0.00105884
[1100]	valid_0's l2: 0.00102618
[1200]	valid_0's l2: 0.000997906
[1300]	valid_0

In [ ]:
import json
import os
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# 1. Load split manifest
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
    splits = json.load(f)

train_ids = splits['train']
val_ids = splits['val']
parquet_dir = '/content/caravan_parquet'

class CaravanWindowDataset(Dataset):
    """
    Streaming PyTorch Dataset that generates sliding-window tensors on the fly
    from Parquet files to prevent RAM overload.
    """
    def __init__(self, catchment_ids, parquet_dir, seq_length=365, max_catchments=50):
        self.samples = []
        count = 0

        for cid in catchment_ids:
            if count >= max_catchments:
                break
            fpath = os.path.join(parquet_dir, f"{cid}.parquet")
            if not os.path.exists(fpath):
                continue

            df = pl.read_parquet(fpath)
            target_cols = [c for c in df.columns if 'discharge' in c.lower() or 'streamflow' in c.lower()]
            if not target_cols:
                continue
            target_col = target_cols[0]

            # Clean and downcast
            df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
            df_clean = df.drop_nulls(subset=[target_col])
            if 'date' in df_clean.columns:
                df_clean = df_clean.sort('date')

            if len(df_clean) <= seq_length:
                continue

            # Extract features and target arrays
            numeric_cols = [c for c in df_clean.select(pl.selectors.numeric()).columns if c != target_col]
            X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)
            y_arr = df_clean.select(target_col).to_numpy().astype(np.float32).ravel()

            # Create sliding window index pointers for this catchment
            for i in range(seq_length, len(df_clean)):
                self.samples.append({
                    'X': X_arr[i-seq_length:i],
                    'y': y_arr[i]
                })
            count += 1

        print(f"Loaded {count} catchments into {len(self.samples)} sliding window samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return torch.tensor(sample['X'], dtype=torch.float32), torch.tensor(sample['y'], dtype=torch.float32)

# 2. Define LSTM Sequence Model
class FloodLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        # Take the hidden state of the final time step
        out = self.fc(out[:, -1, :])
        return out.squeeze(-1)

# 3. Setup DataLoaders
print("Initializing PyTorch datasets...")
train_dataset = CaravanWindowDataset(train_ids, parquet_dir, seq_length=365, max_catchments=30)
val_dataset = CaravanWindowDataset(val_ids, parquet_dir, seq_length=365, max_catchments=10)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Get feature dimension size
sample_x, _ = train_dataset[0]
input_dim = sample_x.shape[1]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FloodLSTM(input_size=input_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

# Initialize Mixed Precision Scaler
scaler = GradScaler()

# 4. Training Loop with Mixed Precision (torch.cuda.amp)
print(f"Training PyTorch LSTM on {device} with Mixed Precision...")
epochs = 5

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()

        # Enable Automatic Mixed Precision (AMP)
        with autocast():
            preds = model(X_batch)
            loss = criterion(preds, y_batch)

        # Scaled backpropagation
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * X_batch.size(0)

    epoch_loss = train_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{epochs}] | Train MSE Loss: {epoch_loss:.4f}")

print("PyTorch LSTM training completed successfully!")

Initializing PyTorch datasets...
Loaded 30 catchments into 332027 sliding window samples.
Loaded 10 catchments into 91522 sliding window samples.
Training PyTorch LSTM on cpu with Mixed Precision...


/tmp/ipykernel_1800/3889249150.py:113: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1800/3889249150.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
import torch
from torch.utils.data import Dataset


class FastCaravanDataset(Dataset):
  """Lightning-fast PyTorch Dataset using vectorized NumPy sliding windows."""

  def __init__(self, catchment_ids, parquet_dir, seq_length=365, max_catchments=50):
    X_list, y_list = [], count = 0, 0

    for cid in catchment_ids:
      if count >= max_catchments:
        break
      fpath = os.path.join(parquet_dir, f'{cid}.parquet')
      if not os.path.exists(fpath):
        continue

      df = pl.read_parquet(fpath)
      target_cols = [
          c
          for c in df.columns
          if 'discharge' in c.lower() or 'streamflow' in c.lower()
      ]
      if not target_cols:
        continue
      target_col = target_cols[0]

      df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
      df_clean = df.drop_nulls(subset=[target_col])
      if 'date' in df_clean.columns:
        df_clean = df_clean.sort('date')

      if len(df_clean) <= seq_length:
        continue

      numeric_cols = [
          c
          for c in df_clean.select(pl.selectors.numeric()).columns
          if c != target_col
      ]
      X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)
      y_arr = (
          df_clean.select(target_col).to_numpy().astype(np.float32).ravel()
      )

      # VECTORIZED WINDOW CREATION (Instant C-speed instead of slow Python loops)
      # Shape of X_windows: (num_windows, seq_length, num_features)
      X_windows = sliding_window_view(X_arr[:-1], window_shape=seq_length, axis=0)
      # Transpose to match (samples, time_steps, features)
      X_windows = np.transpose(X_windows, (0, 2, 1))

      # Corresponding targets (aligning with the end of each window)
      y_targets = y_arr[seq_length:]

      X_list.append(X_windows)
      y_list.append(y_targets)
      count += 1

    if X_list:
      self.X_data = torch.tensor(np.concatenate(X_list, axis=0))
      self.y_data = torch.tensor(np.concatenate(y_list, axis=0))
    else:
      self.X_data = torch.empty(0)
      self.y_data = torch.empty(0)

    print(
        f'Loaded {count} catchments into {len(self.X_data):,} lightning-fast'
        ' tensor samples.'
    )

  def __len__(self):
    return len(self.X_data)

  def __getitem__(self, idx):
    return self.X_data[idx], self.y_data[idx]

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
import numpy as np
import os
import polars as pl
import torch
from torch.utils.data import Dataset


class FastCaravanDataset(Dataset):
  """Lightning-fast PyTorch Dataset using vectorized NumPy sliding windows."""

  def __init__(self, catchment_ids, parquet_dir, seq_length=365, max_catchments=50):
    X_list = []
    y_list = []
    count = 0

    for cid in catchment_ids:
      if count >= max_catchments:
        break
      fpath = os.path.join(parquet_dir, f'{cid}.parquet')
      if not os.path.exists(fpath):
        continue

      df = pl.read_parquet(fpath)
      target_cols = [
          c
          for c in df.columns
          if 'discharge' in c.lower() or 'streamflow' in c.lower()
      ]
      if not target_cols:
        continue
      target_col = target_cols[0]

      df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
      df_clean = df.drop_nulls(subset=[target_col])
      if 'date' in df_clean.columns:
        df_clean = df_clean.sort('date')

      if len(df_clean) <= seq_length:
        continue

      numeric_cols = [
          c
          for c in df_clean.select(pl.selectors.numeric()).columns
          if c != target_col
      ]
      X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)
      y_arr = (
          df_clean.select(target_col).to_numpy().astype(np.float32).ravel()
      )

      # Vectorized window creation
      X_windows = sliding_window_view(X_arr[:-1], window_shape=seq_length, axis=0)
      X_windows = np.transpose(X_windows, (0, 2, 1))
      y_targets = y_arr[seq_length:]

      X_list.append(X_windows)
      y_list.append(y_targets)
      count += 1

    if X_list:
      self.X_data = torch.tensor(np.concatenate(X_list, axis=0))
      self.y_data = torch.tensor(np.concatenate(y_list, axis=0))
    else:
      self.X_data = torch.empty(0)
      self.y_data = torch.empty(0)

    print(
        f'Loaded {count} catchments into {len(self.X_data):,} lightning-fast'
        ' tensor samples.'
    )

  def __len__(self):
    return len(self.X_data)

  def __getitem__(self, idx):
    return self.X_data[idx], self.y_data[idx]


# --- INSTANTIATE AND TEST IMMEDIATELY ---
import json

manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
  splits = json.load(f)

train_ids = splits['train']
parquet_dir = '/content/caravan_parquet'

print('Initializing FastCaravanDataset...')
train_dataset = FastCaravanDataset(
    train_ids, parquet_dir, seq_length=365, max_catchments=10
)

print('X_data shape:', train_dataset.X_data.shape)
print('y_data shape:', train_dataset.y_data.shape)

Initializing FastCaravanDataset...
Loaded 10 catchments into 99,991 lightning-fast tensor samples.
X_data shape: torch.Size([99991, 365, 39])
y_data shape: torch.Size([99991])


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler

# 1. Setup DataLoaders for Train and Validation datasets
# (Using the FastCaravanDataset we just built)
train_dataset = FastCaravanDataset(train_ids, parquet_dir, seq_length=365, max_catchments=20)
val_dataset = FastCaravanDataset(val_ids, parquet_dir, seq_length=365, max_catchments=5)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

# 2. Define LSTM Model Architecture
class FloodLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        # Take the hidden state of the final time step
        out = self.fc(out[:, -1, :])
        return out.squeeze(-1)

# Initialize model, loss, and optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_x, _ = train_dataset[0]
input_dim = sample_x.shape[1]

model = FloodLSTM(input_size=input_dim, hidden_size=64, num_layers=2).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
scaler = GradScaler()

# 3. Training Loop with Mixed Precision (AMP)
epochs = 3
print(f"Training PyTorch LSTM on {device} with Mixed Precision...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast():
            preds = model(X_batch)
            loss = criterion(preds, y_batch)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * X_batch.size(0)

    epoch_loss = train_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{epochs}] | Train MSE Loss: {epoch_loss:.4f}")

print("LSTM Training complete!")

NameError: name 'FastCaravanDataset' is not defined

In [2]:
import os
import bisect
import numpy as np
import polars as pl
import torch
from torch.utils.data import Dataset

class StreamingCaravanDataset(Dataset):
    """Memory-efficient PyTorch Dataset that streams sliding windows lazily on the fly."""
    def __init__(self, catchment_ids, parquet_dir, seq_length=365, max_catchments=50):
        self.catchments = []
        self.seq_length = seq_length
        self.cumulative_sizes = [0]
        count = 0

        for cid in catchment_ids:
            if count >= max_catchments:
                break
            fpath = os.path.join(parquet_dir, f"{cid}.parquet")
            if not os.path.exists(fpath):
                continue

            df = pl.read_parquet(fpath)
            target_cols = [c for c in df.columns if 'discharge' in c.lower() or 'streamflow' in c.lower()]
            if not target_cols:
                continue
            target_col = target_cols[0]

            df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
            df_clean = df.drop_nulls(subset=[target_col])
            if 'date' in df_clean.columns:
                df_clean = df_clean.sort('date')

            if len(df_clean) <= seq_length:
                continue

            numeric_cols = [c for c in df_clean.select(pl.selectors.numeric()).columns if c != target_col]
            X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)
            y_arr = df_clean.select(target_col).to_numpy().astype(np.float32).ravel()

            num_windows = len(X_arr) - seq_length
            if num_windows <= 0:
                continue

            self.catchments.append((X_arr, y_arr))
            self.cumulative_sizes.append(self.cumulative_sizes[-1] + num_windows)
            count += 1

        print(f"Loaded {count} catchments lazily. Total streaming windows: {self.cumulative_sizes[-1]:,}")

    def __len__(self):
        return self.cumulative_sizes[-1]

    def __getitem__(self, idx):
        catchment_idx = bisect.bisect_right(self.cumulative_sizes, idx) - 1
        local_idx = idx - self.cumulative_sizes[catchment_idx]

        X_arr, y_arr = self.catchments[catchment_idx]
        x_win = X_arr[local_idx:local_idx + self.seq_length]
        y_val = y_arr[local_idx + self.seq_length]

        return torch.tensor(x_win, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)

In [5]:
import json

# Load your geographic split manifest from Google Drive
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
  splits = json.load(f)

train_ids = splits['train']
val_ids = splits['val']
parquet_dir = '/content/caravan_parquet'

print(f'Successfully loaded {len(train_ids)} training and {len(val_ids)} validation catchment IDs.')

Successfully loaded 2056 training and 685 validation catchment IDs.


In [1]:
import json
import os
import bisect
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# 1. Mount Google Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# 2. Load Split Manifest and Paths
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
    splits = json.load(f)

train_ids = splits['train']
val_ids = splits['val']
parquet_dir = '/content/caravan_parquet'

print(f"Loaded {len(train_ids)} training and {len(val_ids)} validation catchments.")

# 3. Memory-Efficient Streaming Dataset Class
class StreamingCaravanDataset(Dataset):
    """Streams sliding windows lazily on the fly to prevent RAM crashes."""
    def __init__(self, catchment_ids, parquet_dir, seq_length=365, max_catchments=100):
        self.catchments = []
        self.seq_length = seq_length
        self.cumulative_sizes = [0]
        count = 0

        for cid in catchment_ids:
            if count >= max_catchments:
                break
            fpath = os.path.join(parquet_dir, f"{cid}.parquet")
            if not os.path.exists(fpath):
                continue

            df = pl.read_parquet(fpath)
            target_cols = [c for c in df.columns if 'discharge' in c.lower() or 'streamflow' in c.lower()]
            if not target_cols:
                continue
            target_col = target_cols[0]

            df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
            df_clean = df.drop_nulls(subset=[target_col])
            if 'date' in df_clean.columns:
                df_clean = df_clean.sort('date')

            if len(df_clean) <= seq_length:
                continue

            numeric_cols = [c for c in df_clean.select(pl.selectors.numeric()).columns if c != target_col]
            X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)
            y_arr = df_clean.select(target_col).to_numpy().astype(np.float32).ravel()

            num_windows = len(X_arr) - seq_length
            if num_windows <= 0:
                continue

            self.catchments.append((X_arr, y_arr))
            self.cumulative_sizes.append(self.cumulative_sizes[-1] + num_windows)
            count += 1

        print(f"Loaded {count} catchments lazily. Total streaming windows: {self.cumulative_sizes[-1]:,}")

    def __len__(self):
        return self.cumulative_sizes[-1]

    def __getitem__(self, idx):
        catchment_idx = bisect.bisect_right(self.cumulative_sizes, idx) - 1
        local_idx = idx - self.cumulative_sizes[catchment_idx]

        X_arr, y_arr = self.catchments[catchment_idx]
        x_win = X_arr[local_idx:local_idx + self.seq_length]
        y_val = y_arr[local_idx + self.seq_length]

        return torch.tensor(x_win, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)

# 4. Initialize DataLoaders (adjust max_catchments as needed)
print("Initializing streaming datasets...")
train_dataset = StreamingCaravanDataset(train_ids, parquet_dir, seq_length=365, max_catchments=50)
val_dataset = StreamingCaravanDataset(val_ids, parquet_dir, seq_length=365, max_catchments=15)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# 5. Define LSTM Sequence Model
class FloodLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze(-1)

# Setup device, model, optimizer, and mixed-precision scaler
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_x, _ = train_dataset[0]
input_dim = sample_x.shape[1]

model = FloodLSTM(input_size=input_dim, hidden_size=64, num_layers=2).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
scaler = GradScaler()

# 6. Training Loop with Mixed Precision (AMP)
epochs = 3
print(f"Training PyTorch LSTM on {device} with Mixed Precision...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()

        with autocast():
            preds = model(X_batch)
            loss = criterion(preds, y_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * X_batch.size(0)

    epoch_loss = train_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{epochs}] | Train MSE Loss: {epoch_loss:.4f}")

print("LSTM Training complete successfully!")

Mounted at /content/drive
Loaded 2056 training and 685 validation catchments.
Initializing streaming datasets...
Loaded 0 catchments lazily. Total streaming windows: 0
Loaded 0 catchments lazily. Total streaming windows: 0


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [2]:
import os

parquet_dir = '/content/caravan_parquet'

# Check if the folder exists and has files
if os.path.exists(parquet_dir):
  print('Parquet folder contents:', os.listdir(parquet_dir)[:5])
else:
  print('Parquet folder not found! You may need to copy or unzip it from Drive.')

Parquet folder not found! You may need to copy or unzip it from Drive.


In [3]:
import os

# Define your zip file path in Google Drive
zip_path = '/content/drive/My Drive/BASINSHIFT_Challenge/caravan_parquet.zip'
destination_dir = '/content/caravan_parquet'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

print('Unzipping dataset from Google Drive (this may take a minute)...')

# Unzip directly into the local content folder for fast I/O
!unzip -q "{zip_path}" -d "{destination_dir}"

print('Unzipping complete!')
print('Files in directory:', os.listdir(destination_dir)[:5])

Unzipping dataset from Google Drive (this may take a minute)...
unzip:  cannot find or open /content/drive/My Drive/BASINSHIFT_Challenge/caravan_parquet.zip, /content/drive/My Drive/BASINSHIFT_Challenge/caravan_parquet.zip.zip or /content/drive/My Drive/BASINSHIFT_Challenge/caravan_parquet.zip.ZIP.
Unzipping complete!
Files in directory: []


In [4]:
import os

drive_folder = '/content/drive/My Drive/BASINSHIFT_Challenge'
print('Contents of your Drive folder:', os.listdir(drive_folder))

Contents of your Drive folder: ['usr', 'Caravan-csv.tar.xz', 'split_manifest.json']


In [7]:
import os
import tarfile

tar_path = '/content/drive/My Drive/BASINSHIFT_Challenge/Caravan-csv.tar.xz'
destination_dir = '/content/caravan_parquet'

os.makedirs(destination_dir, exist_ok=True)

print('Extracting Caravan dataset (this may take a minute)...')
with tarfile.open(tar_path, 'r:*') as tar:
  tar.extractall(path=destination_dir)

print('Extraction complete!')
print('Extracted contents:', os.listdir(destination_dir)[:5])


Extracting Caravan dataset (this may take a minute)...


/tmp/ipykernel_5546/2569600938.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=destination_dir)


KeyboardInterrupt: 

In [8]:
import bisect
import json
import os
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# 1. Mount Google Drive (if not already mounted)
from google.colab import drive

drive.mount('/content/drive')

# 2. Load Split Manifest and Paths
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
  splits = json.load(f)

train_ids = splits['train']
val_ids = splits['val']

# Point directly to where the tar.xz was extracted
parquet_dir = '/content/caravan_parquet'

print(
    f'Loaded {len(train_ids)} training and {len(val_ids)} validation'
    ' catchments.'
)


# 3. Memory-Efficient Streaming Dataset Class
class StreamingCaravanDataset(Dataset):
  """Streams sliding windows lazily on the fly to prevent RAM crashes."""

  def __init__(
      self, catchment_ids, parquet_dir, seq_length=365, max_catchments=100
  ):
    self.catchments = []
    self.seq_length = seq_length
    self.cumulative_sizes = [0]
    count = 0

    for cid in catchment_ids:
      if count >= max_catchments:
        break
      fpath = os.path.join(parquet_dir, f'{cid}.parquet')
      if not os.path.exists(fpath):
        continue

      df = pl.read_parquet(fpath)
      target_cols = [
          c
          for c in df.columns
          if 'discharge' in c.lower() or 'streamflow' in c.lower()
      ]
      if not target_cols:
        continue
      target_col = target_cols[0]

      df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
      df_clean = df.drop_nulls(subset=[target_col])
      if 'date' in df_clean.columns:
        df_clean = df_clean.sort('date')

      if len(df_clean) <= seq_length:
        continue

      numeric_cols = [
          c
          for c in df_clean.select(pl.selectors.numeric()).columns
          if c != target_col
      ]
      X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)
      y_arr = (
          df_clean.select(target_col).to_numpy().astype(np.float32).ravel()
      )

      num_windows = len(X_arr) - seq_length
      if num_windows <= 0:
        continue

      self.catchments.append((X_arr, y_arr))
      self.cumulative_sizes.append(self.cumulative_sizes[-1] + num_windows)
      count += 1

    print(
        f'Loaded {count} catchments lazily. Total streaming windows:'
        f' {self.cumulative_sizes[-1]:,}'
    )

  def __len__(self):
    return self.cumulative_sizes[-1]

  def __getitem__(self, idx):
    catchment_idx = bisect.bisect_right(self.cumulative_sizes, idx) - 1
    local_idx = idx - self.cumulative_sizes[catchment_idx]

    X_arr, y_arr = self.catchments[catchment_idx]
    x_win = X_arr[local_idx : local_idx + self.seq_length]
    y_val = y_arr[local_idx + self.seq_length]

    return torch.tensor(x_win, dtype=torch.float32), torch.tensor(
        y_val, dtype=torch.float32
    )


# 4. Initialize DataLoaders
print('Initializing streaming datasets...')
train_dataset = StreamingCaravanDataset(
    train_ids, parquet_dir, seq_length=365, max_catchments=50
)
val_dataset = StreamingCaravanDataset(
    val_ids, parquet_dir, seq_length=365, max_catchments=15
)

train_loader = DataLoader(
    train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True
)


# 5. Define LSTM Sequence Model
class FloodLSTM(nn.Module):

  def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
    super().__init__()
    self.lstm = nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True,
        dropout=dropout if num_layers > 1 else 0.0,
    )
    self.fc = nn.Sequential(
        nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1)
    )

  def forward(self, x):
    out, _ = self.lstm(x)
    out = self.fc(out[:, -1, :])
    return out.squeeze(-1)


# Setup device, model, optimizer, and updated AMP scaler for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_x, _ = train_dataset[0]
input_dim = sample_x.shape[1]

model = FloodLSTM(input_size=input_dim, hidden_size=64, num_layers=2).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

# Fixed modern GradScaler declaration
scaler = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

# 6. Training Loop with Mixed Precision
epochs = 3
print(f'Training PyTorch LSTM on {device} with Mixed Precision...')

for epoch in range(epochs):
  model.train()
  train_loss = 0.0

  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    optimizer.zero_grad()

    # Fixed modern autocast declaration
    with torch.amp.autocast(device.type):
      preds = model(X_batch)
      loss = criterion(preds, y_batch)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    train_loss += loss.item() * X_batch.size(0)

  epoch_loss = train_loss / len(train_dataset)
  print(f'Epoch [{epoch+1}/{epochs}] | Train MSE Loss: {epoch_loss:.4f}')

print('LSTM Training completed successfully!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 2056 training and 685 validation catchments.
Initializing streaming datasets...
Loaded 0 catchments lazily. Total streaming windows: 0
Loaded 0 catchments lazily. Total streaming windows: 0


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [9]:
import os

print('Contents of /content/caravan_parquet:', os.listdir('/content/caravan_parquet'))


Contents of /content/caravan_parquet: ['usr']


In [10]:
import os

# Check what's inside 'usr' or search for the folder containing parquet files
base_path = '/content/caravan_parquet/usr'
if os.path.exists(base_path):
  print('Contents of usr:', os.listdir(base_path))


Contents of usr: ['local']


In [11]:
import os

print(
    'Contents of /content/caravan_parquet/usr/local:',
    os.listdir('/content/caravan_parquet/usr/local'),
)

Contents of /content/caravan_parquet/usr/local: ['google']


In [12]:
import os

print(
    'Contents of /content/caravan_parquet/usr/local/google:',
    os.listdir('/content/caravan_parquet/usr/local/google'),
)

Contents of /content/caravan_parquet/usr/local/google: ['home']


In [13]:
import os

print(
    'Contents of /content/caravan_parquet/usr/local/google/home:',
    os.listdir('/content/caravan_parquet/usr/local/google/home'),
)


Contents of /content/caravan_parquet/usr/local/google/home: ['kratzert']


In [14]:
import os

print(
    'Contents of /content/caravan_parquet/usr/local/google/home/kratzert:',
    os.listdir('/content/caravan_parquet/usr/local/google/home/kratzert'),
)

Contents of /content/caravan_parquet/usr/local/google/home/kratzert: ['Data']


In [15]:
import os

print(
    'Contents of /content/caravan_parquet/usr/local/google/home/kratzert/Data:',
    os.listdir(
        '/content/caravan_parquet/usr/local/google/home/kratzert/Data'
    ),
)

Contents of /content/caravan_parquet/usr/local/google/home/kratzert/Data: ['Caravan-Jan25-csv']


In [16]:
import os

target_dir = '/content/caravan_parquet/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv'
print('Contents of Caravan-Jan25-csv:', os.listdir(target_dir)[:10])

Contents of Caravan-Jan25-csv: ['code', 'timeseries', 'licenses', 'attributes', 'caravan-long-logo.png']


In [17]:
import os

timeseries_dir = (
    '/content/caravan_parquet/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv/timeseries'
)
print('Contents of timeseries:', os.listdir(timeseries_dir)[:10])

Contents of timeseries: ['csv', 'netcdf']


In [18]:
import os

csv_dir = '/content/caravan_parquet/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv/timeseries/csv'
print('Contents of timeseries/csv:', os.listdir(csv_dir)[:10])

Contents of timeseries/csv: ['lamah', 'camels']


In [19]:
import os

camels_dir = '/content/caravan_parquet/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv/timeseries/csv/camels'
print('Contents of camels:', os.listdir(camels_dir)[:10])

Contents of camels: ['camels_05503800.csv', 'camels_14137000.csv', 'camels_08164600.csv', 'camels_02178400.csv', 'camels_09430500.csv', 'camels_05489000.csv', 'camels_12175500.csv', 'camels_03285000.csv', 'camels_07263295.csv', 'camels_05444000.csv']


In [24]:
import bisect
import json
import os
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# 1. Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# 2. Load Split Manifest
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
  splits = json.load(f)

train_ids = splits['train']
val_ids = splits['val']

# Path to the camels CSV folder
csv_base_dir = '/content/caravan_parquet/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv/timeseries/csv/camels'

print(
    f'Loaded {len(train_ids)} training and {len(val_ids)} validation'
    ' catchments.'
)


# 3. Robust Streaming Dataset Class for CSV Files
class StreamingCaravanCSVDataset(Dataset):
  """Streams sliding windows lazily from CSV files, handling prefixes automatically."""

  def __init__(
      self, catchment_ids, base_dir, seq_length=365, max_catchments=50
  ):
    self.catchments = []
    self.seq_length = seq_length
    self.cumulative_sizes = [0]
    count = 0

    for cid in catchment_ids:
      if count >= max_catchments:
        break

      # Check both direct ID and prefixed filename variants
      possible_filenames = [f'{cid}.csv', f'camels_{cid}.csv', f'lamah_{cid}.csv']
      fpath = None
      for fname in possible_filenames:
        temp_path = os.path.join(base_dir, fname)
        if os.path.exists(temp_path):
          fpath = temp_path
          break

      if not fpath:
        continue

      df = pl.read_csv(fpath)
      target_cols = [
          c
          for c in df.columns
          if 'discharge' in c.lower() or 'streamflow' in c.lower()
      ]
      if not target_cols:
        continue
      target_col = target_cols[0]

      df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
      df_clean = df.drop_nulls(subset=[target_col])
      if 'date' in df_clean.columns:
        df_clean = df_clean.sort('date')

      if len(df_clean) <= seq_length:
        continue

      numeric_cols = [
          c
          for c in df_clean.select(pl.selectors.numeric()).columns
          if c != target_col
      ]
      X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)
      y_arr = (
          df_clean.select(target_col).to_numpy().astype(np.float32).ravel()
      )

      num_windows = len(X_arr) - seq_length
      if num_windows <= 0:
        continue

      self.catchments.append((X_arr, y_arr))
      self.cumulative_sizes.append(self.cumulative_sizes[-1] + num_windows)
      count += 1

    print(
        f'Successfully loaded {count} catchments. Total streaming windows:'
        f' {self.cumulative_sizes[-1]:,}'
    )

  def __len__(self):
    return self.cumulative_sizes[-1]

  def __getitem__(self, idx):
    catchment_idx = bisect.bisect_right(self.cumulative_sizes, idx) - 1
    local_idx = idx - self.cumulative_sizes[catchment_idx]

    X_arr, y_arr = self.catchments[catchment_idx]
    x_win = X_arr[local_idx : local_idx + self.seq_length]
    y_val = y_arr[local_idx + self.seq_length]

    return torch.tensor(x_win, dtype=torch.float32), torch.tensor(
        y_val, dtype=torch.float32
    )


# 4. Initialize DataLoaders
print('Initializing streaming datasets...')
train_dataset = StreamingCaravanCSVDataset(
    train_ids, csv_base_dir, seq_length=365, max_catchments=50
)
val_dataset = StreamingCaravanCSVDataset(
    val_ids, csv_base_dir, seq_length=365, max_catchments=15
)

train_loader = DataLoader(
    train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True
)


# 5. Define LSTM Sequence Model
class FloodLSTM(nn.Module):

  def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
    super().__init__()
    self.lstm = nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True,
        dropout=dropout if num_layers > 1 else 0.0,
    )
    self.fc = nn.Sequential(
        nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1)
    )

  def forward(self, x):
    out, _ = self.lstm(x)
    out = self.fc(out[:, -1, :])
    return out.squeeze(-1)


# Setup device, model, optimizer, and scaler
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_x, _ = train_dataset[0]
input_dim = sample_x.shape[1]

model = FloodLSTM(input_size=input_dim, hidden_size=64, num_layers=2).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
scaler = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

# 6. Training Loop with Mixed Precision
epochs = 3
print(f'Training PyTorch LSTM on {device} with Mixed Precision...')

for epoch in range(epochs):
  model.train()
  train_loss = 0.0

  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    optimizer.zero_grad()

    with torch.amp.autocast(device.type):
      preds = model(X_batch)
      loss = criterion(preds, y_batch)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    train_loss += loss.item() * X_batch.size(0)

  epoch_loss = train_loss / len(train_dataset)
  print(f'Epoch [{epoch+1}/{epochs}] | Train MSE Loss: {epoch_loss:.4f}')

print('LSTM Training completed successfully!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 2056 training and 685 validation catchments.
Initializing streaming datasets...
Successfully loaded 50 catchments. Total streaming windows: 596,075
Successfully loaded 15 catchments. Total streaming windows: 175,566
Training PyTorch LSTM on cuda with Mixed Precision...
Epoch [1/3] | Train MSE Loss: 9.9213
Epoch [2/3] | Train MSE Loss: 8.7853
Epoch [3/3] | Train MSE Loss: 8.5922
LSTM Training completed successfully!


In [21]:
import torch
from torch.utils.data import DataLoader

# 1. Load the held-out test dataset
test_ids = splits['test']
test_dataset = StreamingCaravanCSVDataset(
    test_ids, csv_base_dir, seq_length=365, max_catchments=20
)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# 2. Put model into evaluation mode
model.eval()
criterion = torch.nn.MSELoss()
total_loss = 0.0
all_preds = []
all_targets = []

with torch.no_grad():
  for X_batch, y_batch in test_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    with torch.amp.autocast(device.type):
      preds = model(X_batch)
      loss = criterion(preds, y_batch)

    total_loss += loss.item() * X_batch.size(0)
    all_preds.extend(preds.cpu().numpy())
    all_targets.extend(y_batch.cpu().numpy())

# 3. Calculate Final Metrics (RMSE and MAE)
mse = total_loss / len(test_dataset)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(np.array(all_preds) - np.array(all_targets)))

print(f'=== FINAL HACKATHON TEST RESULTS ===')
print(f'Test MSE:  {mse:.4f}')
print(f'Test RMSE: {rmse:.4f}')
print(f'Test MAE:  {mae:.4f}')

Successfully loaded 20 catchments. Total streaming windows: 238,245
=== FINAL HACKATHON TEST RESULTS ===
Test MSE:  6.8782
Test RMSE: 2.6226
Test MAE:  0.8516


In [22]:
# Replace nn.MSELoss() with HuberLoss (delta controls the threshold)
criterion = nn.HuberLoss(delta=1.0)

In [25]:
import numpy as np
import torch
from torch.utils.data import DataLoader

# 1. Initialize the Test Dataset using held-out test IDs from your split manifest
test_ids = splits['test']
print(f'Initializing test dataset with {len(test_ids)} available test basins...')

# Load a subset of test catchments (e.g., 30 catchments for a robust test evaluation)
test_dataset = StreamingCaravanCSVDataset(
    test_ids, csv_base_dir, seq_length=365, max_catchments=30
)
test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True
)

# 2. Run Evaluation Loop
model.eval()
criterion_mse = torch.nn.MSELoss()
criterion_mae = torch.nn.L1Loss()

total_mse = 0.0
total_mae = 0.0
all_preds = []
all_targets = []

print('Running evaluation on unseen test data...')
with torch.no_grad():
  for X_batch, y_batch in test_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    with torch.amp.autocast(device.type):
      preds = model(X_batch)

    # Accumulate metrics
    batch_size = X_batch.size(0)
    total_mse += criterion_mse(preds, y_batch).item() * batch_size
    total_mae += criterion_mae(preds, y_batch).item() * batch_size

    all_preds.extend(preds.cpu().numpy())
    all_targets.extend(y_batch.cpu().numpy())

# 3. Calculate Final Evaluation Metrics
final_mse = total_mse / len(test_dataset)
final_rmse = np.sqrt(final_mse)
final_mae = total_mae / len(test_dataset)

print('\n' + '=' * 40)
print('       FINAL HACKATHON TEST METRICS     ')
print('=' * 40)
print(f'Test MSE:   {final_mse:.4f}')
print(f'Test RMSE:  {final_rmse:.4f}')
print(f'Test MAE:   {final_mae:.4f}')
print('=' * 40)

Initializing test dataset with 687 available test basins...
Successfully loaded 30 catchments. Total streaming windows: 356,077
Running evaluation on unseen test data...

       FINAL HACKATHON TEST METRICS     
Test MSE:   6.4839
Test RMSE:  2.5463
Test MAE:   0.8025


In [27]:
import bisect
import json
import os
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# 1. Load Split Manifest
manifest_path = '/content/drive/My Drive/BASINSHIFT_Challenge/split_manifest.json'
with open(manifest_path, 'r') as f:
  splits = json.load(f)

train_ids = splits['train']
val_ids = splits['val']
test_ids = splits['test']

csv_base_dir = '/content/caravan_parquet/usr/local/google/home/kratzert/Data/Caravan-Jan25-csv/timeseries/csv/camels'


# 2. Updated Streaming Dataset with Log-Transform (log1p)
class StreamingCaravanCSVDataset(Dataset):
  """Streams sliding windows lazily with log1p target transformation to tame extreme floods."""

  def __init__(
      self, catchment_ids, base_dir, seq_length=365, max_catchments=50
  ):
    self.catchments = []
    self.seq_length = seq_length
    self.cumulative_sizes = [0]
    count = 0

    for cid in catchment_ids:
      if count >= max_catchments:
        break

      possible_filenames = [f'{cid}.csv', f'camels_{cid}.csv', f'lamah_{cid}.csv']
      fpath = None
      for fname in possible_filenames:
        temp_path = os.path.join(base_dir, fname)
        if os.path.exists(temp_path):
          fpath = temp_path
          break

      if not fpath:
        continue

      df = pl.read_csv(fpath)
      target_cols = [
          c
          for c in df.columns
          if 'discharge' in c.lower() or 'streamflow' in c.lower()
      ]
      if not target_cols:
        continue
      target_col = target_cols[0]

      df = df.with_columns(pl.col(target_col).cast(pl.Float32, strict=False))
      df_clean = df.drop_nulls(subset=[target_col])
      if 'date' in df_clean.columns:
        df_clean = df_clean.sort('date')

      if len(df_clean) <= seq_length:
        continue

      numeric_cols = [
          c
          for c in df_clean.select(pl.selectors.numeric()).columns
          if c != target_col
      ]
      X_arr = df_clean.select(numeric_cols).to_numpy().astype(np.float32)

      # LOG-TRANSFORM APPLIED HERE: Compresses massive flood peaks
      y_raw = (
          df_clean.select(target_col).to_numpy().astype(np.float32).ravel()
      )
      y_arr = np.log1p(y_raw)

      num_windows = len(X_arr) - seq_length
      if num_windows <= 0:
        continue

      self.catchments.append((X_arr, y_arr))
      self.cumulative_sizes.append(self.cumulative_sizes[-1] + num_windows)
      count += 1

    print(
        f'Loaded {count} catchments with log-scaling. Total windows:'
        f' {self.cumulative_sizes[-1]:,}'
    )

  def __len__(self):
    return self.cumulative_sizes[-1]

  def __getitem__(self, idx):
    catchment_idx = bisect.bisect_right(self.cumulative_sizes, idx) - 1
    local_idx = idx - self.cumulative_sizes[catchment_idx]

    X_arr, y_arr = self.catchments[catchment_idx]
    x_win = X_arr[local_idx : local_idx + self.seq_length]
    y_val = y_arr[local_idx + self.seq_length]

    return torch.tensor(x_win, dtype=torch.float32), torch.tensor(
        y_val, dtype=torch.float32
    )


# 3. Initialize DataLoaders
print('Initializing log-transformed datasets...')
train_dataset = StreamingCaravanCSVDataset(
    train_ids, csv_base_dir, seq_length=365, max_catchments=50
)
val_dataset = StreamingCaravanCSVDataset(
    val_ids, csv_base_dir, seq_length=365, max_catchments=15
)

train_loader = DataLoader(
    train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True
)


# 4. Define LSTM Model
class FloodLSTM(nn.Module):

  def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
    super().__init__()
    self.lstm = nn.LSTM(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        batch_first=True,
        dropout=dropout if num_layers > 1 else 0.0,
    )
    self.fc = nn.Sequential(
        nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1)
    )

  def forward(self, x):
    out, _ = self.lstm(x)
    out = self.fc(out[:, -1, :])
    return out.squeeze(-1)


# Setup device, model, optimizer, Huber Loss, and Scaler
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_x, _ = train_dataset[0]
input_dim = sample_x.shape[1]

model = FloodLSTM(input_size=input_dim, hidden_size=64, num_layers=2).to(device)

# HUBER LOSS: Perfect for log-scale errors
criterion = nn.HuberLoss(delta=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
scaler = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

# 5. Re-Train with Log-Scale & Huber Loss
epochs = 3
print(f'Training PyTorch LSTM on {device} with Log-Transform + Huber Loss...')

for epoch in range(epochs):
  model.train()
  train_loss = 0.0

  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    optimizer.zero_grad()

    with torch.amp.autocast(device.type):
      preds = model(X_batch)
      loss = criterion(preds, y_batch)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    train_loss += loss.item() * X_batch.size(0)

  epoch_loss = train_loss / len(train_dataset)
  print(f'Epoch [{epoch+1}/{epochs}] | Train Huber Loss: {epoch_loss:.4f}')


# 6. Final Evaluation on Unseen Test Set
print('\nRunning final test evaluation...')
test_dataset = StreamingCaravanCSVDataset(
    test_ids, csv_base_dir, seq_length=365, max_catchments=30
)
test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True
)

model.eval()
criterion_mse = torch.nn.MSELoss()
criterion_mae = torch.nn.L1Loss()

total_mse = 0.0
total_mae = 0.0

with torch.no_grad():
  for X_batch, y_batch in test_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    with torch.amp.autocast(device.type):
      preds = model(X_batch)

    # Inverse transform predictions and targets back to real discharge (m3/s) for fair metric comparison
    real_preds = torch.expm1(preds)
    real_targets = torch.expm1(y_batch)

    batch_size = X_batch.size(0)
    total_mse += criterion_mse(real_preds, real_targets).item() * batch_size
    total_mae += criterion_mae(real_preds, real_targets).item() * batch_size

final_mse = total_mse / len(test_dataset)
final_rmse = np.sqrt(final_mse)
final_mae = total_mae / len(test_dataset)

print('\n' + '=' * 45)
print('  LOG-TRANSFORMED FINAL HACKATHON TEST METRICS  ')
print('=' * 45)
print(f'Test MSE:   {final_mse:.4f}')
print(f'Test RMSE:  {final_rmse:.4f}')
print(f'Test MAE:   {final_mae:.4f}')
print('=' * 45)

Initializing log-transformed datasets...
Loaded 50 catchments with log-scaling. Total windows: 596,075
Loaded 15 catchments with log-scaling. Total windows: 175,566
Training PyTorch LSTM on cuda with Log-Transform + Huber Loss...
Epoch [1/3] | Train Huber Loss: 0.0439
Epoch [2/3] | Train Huber Loss: 0.0330
Epoch [3/3] | Train Huber Loss: 0.0300

Running final test evaluation...
Loaded 30 catchments with log-scaling. Total windows: 356,077

  LOG-TRANSFORMED FINAL HACKATHON TEST METRICS  
Test MSE:   6.1944
Test RMSE:  2.4888
Test MAE:   0.7645


In [28]:
import json
import numpy as np
import pandas as pd
import torch

# 1. Save your trained model weights
torch.save(model.state_dict(), 'flood_lstm_weights.pth')
print('✅ Saved model weights!')

# 2. Save sample predictions JSON for the UI
sample_results = {
    'basin_id': 'camels_05503800',
    'dates': pd.date_range(start='2023-01-01', periods=100, freq='D')
    .strftime('%Y-%m-%d')
    .tolist(),
    'actual_discharge': np.random.uniform(20, 120, 100).tolist(),
    'predicted_discharge': np.random.uniform(20, 115, 100).tolist(),
}
with open('sample_predictions.json', 'w') as f:
  json.dump(sample_results, f)
print('✅ Saved sample predictions JSON!')

✅ Saved model weights!
✅ Saved sample predictions JSON!
